In [ ]:
import os
from openai import OpenAI

In [ ]:
openai_client = OpenAI()

In [ ]:
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
response = groq_client.responses.create(
    model="openai/gpt-oss-20b",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [ ]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [ ]:
documents[12]

In [ ]:
!pip install minsearch

In [ ]:
from minsearch import AppendableIndex

In [ ]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
from typing import Any, Dict, List

def search(query: str) -> List[Dict[str, Any]]:
    """
    Search the index for documents matching the given query.

    This function queries the index with a fixed course filter and 
    predefined boost weights for specific fields.

    Args:
        query (str): The search query string.

    Returns:
        List[Dict[str, Any]]: A list of search result objects returned 
        by `index.search()`. Each result is represented as a dictionary.
    """
    boost: Dict[str, float] = {"question": 3.0, "section": 0.5}

    results = index.search(
        query=query,
        filter_dict={"course": "data-engineering-zoomcamp"},
        boost_dict=boost,
        num_results=5,
    )

    return results


def add_entry(question: str, answer: str) -> None:
    """
    Add a new question–answer entry to the index.

    This function constructs a document with the provided question
    and answer, tags it as user-added, and appends it to the index
    under the 'data-engineering-zoomcamp' course.

    Args:
        question (str): The question text to store.
        answer (str): The corresponding answer text.

    Returns:
        None
    """

    doc = {
        'question': question,
        'text': answer,
        'section': 'user added',
        'course': 'data-engineering-zoomcamp'
    }
    index.append(doc)

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
result = search(question)

In [ ]:
import json

In [ ]:
question = 'I just discovered the course. Can I join it now?'

In [ ]:
prompt = f"""
Answer the question from the student using the provided context

<QUESTION>{question}</QUESTION>

<CONTEXT>{json.dumps(result)}</CONTEXT>
"""

search 

- hitrate
- MRR
- NDCG 


Agent

- judges 

In [ ]:
agent_instructions = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

For each question, perfrom at least 3 searches to make sure you explore the topic deep enough.


""".strip()

# agentic RAG

chat_messages = [
    {"role": "developer", "content": agent_instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="gpt-5-nano", #"gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

In [ ]:
response.usage.input_tokens, response.usage.output_tokens

In [ ]:
#$0.050, $0.400 

In [ ]:
!pip install genai-prices

In [ ]:
from genai_prices import Usage, calc_price

In [ ]:
price = calc_price(
    Usage(input_tokens=37, output_tokens=528),
    model_ref='gpt-5-nano',
    provider_id='openai'
)

In [ ]:
price.total_price

In [ ]:
tool_call = response.output[0]
tool_call

In [ ]:
chat_messages.append(tool_call)

In [ ]:
search_result = search(query="Can I join the course now?")

In [ ]:
result_json = json.dumps(search_result, indent=2)

chat_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": result_json,
})

In [ ]:
chat_messages

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

In [ ]:
response.output_text

In [ ]:
chat_messages.append(
    {"role": "user", "content": "but are you sure I can get my certificate?"}
)

In [ ]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)
response.output_text

In [ ]:
chat_messages = [
    {"role": "user", "content": question}
]

response = groq_client.responses.create(
    model="openai/gpt-oss-20b",
    input=chat_messages,
    tools=[search_tool]
)
response.output_text

In [ ]:
response.output

In [ ]:
import json

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)
    f_name = call.name
    f = globals()[f_name]
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = 'I just discovered the course. Can I join it now?'

In [ ]:
agent_instructions = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

If you want to look up the answer, explain why before making the call. Use as many 
keywords from the user question as possible when making first requests.

Make multiple searches (up to 3).

At the end, make a clarifying question based on what you presented and ask if there are 
other areas that the user wants to explore.
""".strip()

# agentic RAG



In [ ]:
cnt = 0

chat_messages = [
    {"role": "developer", "content": agent_instructions},
    {"role": "user", "content": question}
]

while True: # agentic loop / tool call loop
    response = openai_client.responses.create(
        model="gpt-5-nano",#"gpt-4o-mini",
        input=chat_messages,
        tools=[search_tool]
    )
    cnt = cnt + 1
    print('number of requests', cnt)

    chat_messages.extend(response.output)
    
    has_function_calls = False
    
    for message in response.output:
        if message.type == 'message':
            print(message.content[0].text)
    
        if message.type == 'function_call':
            arguments = json.loads(message.arguments)
            f_name = message.name
            print(f_name, arguments)
            results = make_call(message)
            chat_messages.append(results)
    
            has_function_calls = True

    if has_function_calls == False:
        break

    if number_of_tokens > 100000:
        chat_messages.append({'role': 'user', 'message': 'SYSTEM MESSAGE: you have reached the limit, proceed to finishing the response'}
    

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.runners import DisplayingRunnerCallback

    agent_tools = Tools()
    agent_tools.add_tool(search, search_tool)

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.add_tool(add_entry)

In [ ]:
chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-4o-mini')
)

In [ ]:
result = runner.run();

In [ ]:
result.cost

In [ ]:
from pydantic_ai import Agent

In [ ]:
search

In [ ]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=agent_instructions,
    tools=[search, add_entry],
)

In [ ]:
search_agent.run(user_prompt=question)

In [ ]:
messages = []

iteration_nmber = 0

while True:
    user_prompt = input()
    if user_prompt.strip().lower() == 'stop': 
        break

    result = await search_agent.run(
        user_prompt=user_prompt,
        message_history=messages,

    )

    iteration_nmber = iteration_nmber + 1
    print(f'{iteration_nmber=}')
    print(result.output)
    print()

    messages.extend(result.new_messages())

In [ ]:
result.all_messages()

In [ ]:
price = calc_price(
    result.usage(),
    model_ref='gpt-4o-mini',
    provider_id='openai'
)
price.total_price

In [ ]:
from toyaikit.chat.runners import PydanticAIRunner

In [ ]:
pyai_runner = PydanticAIRunner(
    chat_interface=chat_interface,
    agent=search_agent
)

await pyai_runner.run();

In [ ]:
index

In [ ]:
import search_tools

In [ ]:
agent_search_tools = search_tools.SearchTools(index)

In [ ]:
from toyaikit.tools import get_instance_methods

In [ ]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=agent_instructions,
    tools=get_instance_methods(agent_search_tools),
)